# Week 2 — Context-Aware Two-Tower Recommendation Model

Project 3: Context-Aware Neural Recommendation Engine (H&M dataset).

This notebook trains the two-tower model **on the Week 1 outputs** (`MyDrive/hm_project/*.parquet`)
and evaluates it on the **last week** of data against a **popularity baseline**.

**What changed from the first Week 2 version**

| Before | Now |
|---|---|
| Re-read raw CSVs, own cleaning | Uses the Week 1 parquet files and `vocabularies.json` |
| 25,000 rows from Sept 2018 | Last `TRAIN_WEEKS` weeks before 2020-09-16 (millions of rows) |
| Random 90/10 split | Time split: train → validation week → test week (2020-09-16 to 22) |
| Customer ID, club, news, age only | + purchase history, recency, spending, month, day of week, item popularity |
| Every ID has its own embedding, even if never trained | IDs not seen in training use index 0, a shared trained "unknown" embedding |
| LR 0.0001, 3 epochs, batch 256, no normalisation | LR 0.001, up to 10 epochs with early stopping, batch 2048, L2-normalised embeddings |
| No baseline | Compared with the Week 1 popularity fallback (top-12) |

**Before running:** Runtime → Change runtime type → **T4 GPU** (works on CPU too, just slower). Then Runtime → Run all.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, json, time, shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

DATA_PATH = "/content/drive/MyDrive/hm_project"          # Week 1 outputs live here
MODEL_DIR = os.path.join(DATA_PATH, "week2_model")        # trained model goes here (Drive)
ARTIFACT_DIR = "/content/week2_artifacts"                 # small files for the GitHub repo
CUTOFF = pd.Timestamp("2020-09-16")                       # same cutoff as Week 1

# ---- settings you can change ----
TRAIN_WEEKS = 12        # weeks of history before the cutoff to train on (lower to 6 if RAM runs out)
VAL_DAYS = 7            # last week before the cutoff = validation (for early stopping)
BATCH_SIZE = 2048
EPOCHS = 10
PATIENCE = 2            # stop if validation loss doesn't improve for 2 epochs
LEARNING_RATE = 1e-3
EMBEDDING_DIM = 64
ID_DIM = 32
TEMPERATURE = 0.05
ID_DROPOUT = 0.1
TOP_K = 12              # H&M recommends 12 articles
SEED = 42

os.makedirs(ARTIFACT_DIR, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none - training will be slower")

## 2. Load the Week 1 data

- **Train:** purchases from `TRAIN_WEEKS` weeks before the cutoff, except the last week
- **Validation:** the last week before the cutoff (2020-09-09 to 2020-09-15), used for early stopping
- **Test:** `interactions_test.parquet` (2020-09-16 to 2020-09-22), used only for the final evaluation

In [ ]:
interactions = pd.read_parquet(
    os.path.join(DATA_PATH, "interactions_train.parquet"),
    columns=["user_idx", "item_idx", "t_dat", "month", "day_of_week", "days_ago"],
    filters=[("days_ago", "<=", TRAIN_WEEKS * 7)],
)
interactions = interactions[(interactions["user_idx"] > 0) & (interactions["item_idx"] > 0)]
# the same customer buying the same article on the same day counts once
interactions = interactions.drop_duplicates(["user_idx", "item_idx", "t_dat"]).reset_index(drop=True)
for column in ["user_idx", "item_idx", "month", "day_of_week", "days_ago"]:
    interactions[column] = interactions[column].astype("int32")

train_df = (interactions[interactions["days_ago"] > VAL_DAYS]
            .sample(frac=1.0, random_state=SEED).reset_index(drop=True))
val_df = interactions[interactions["days_ago"] <= VAL_DAYS].reset_index(drop=True)

test_df = pd.read_parquet(
    os.path.join(DATA_PATH, "interactions_test.parquet"), columns=["user_idx", "item_idx"]
)
test_df = test_df[(test_df["user_idx"] > 0) & (test_df["item_idx"] > 0)]

print(f"Train:      {len(train_df):>10,} rows  {train_df['t_dat'].min()} -> {train_df['t_dat'].max()}")
print(f"Validation: {len(val_df):>10,} rows  {val_df['t_dat'].min()} -> {val_df['t_dat'].max()}")
print(f"Test:       {len(test_df):>10,} rows  (2020-09-16 -> 2020-09-22)")

In [ ]:
users = pd.read_parquet(os.path.join(DATA_PATH, "user_table.parquet"))
items = pd.read_parquet(os.path.join(DATA_PATH, "item_table.parquet"))
with open(os.path.join(DATA_PATH, "vocabularies.json")) as f:
    vocabularies = json.load(f)
with open(os.path.join(DATA_PATH, "fallback_top12.json")) as f:
    fallback_top12 = json.load(f)

N_USERS = int(users["user_idx"].max())
N_ITEMS = int(items["item_idx"].max())
ITEM_CAT_COLS = ["product_type_name", "colour_group_name", "department_name", "garment_group_name"]
print("Users:", N_USERS, "| Items:", N_ITEMS)


def encode(values, vocab):
    """Category text -> integer code (1..len(vocab)); 0 = unknown."""
    mapping = {value: code for code, value in enumerate(vocab, start=1)}
    return pd.Series(values).map(mapping).fillna(0).astype("int32").to_numpy()


def standardize(frame):
    mean = frame.mean()
    std = frame.std().replace(0, 1).fillna(1)
    return ((frame - mean) / std).fillna(0).astype("float32").to_numpy(), mean, std

### Feature lookup tables
One row per `user_idx` / `item_idx`, so features can be fetched by index. Row 0 is the "unknown" row.

In [ ]:
# ---- user features ----
u = users["user_idx"].to_numpy()
user_club = np.zeros(N_USERS + 1, dtype="int32")
user_news = np.zeros(N_USERS + 1, dtype="int32")
user_club[u] = encode(users["club_member_status"], vocabularies["club_member_status"])
user_news[u] = encode(users["fashion_news_frequency"], vocabularies["fashion_news_frequency"])

user_num_raw = pd.DataFrame({
    "age": users["age"].astype("float32"),
    "age_missing": users["age_missing"].astype("float32"),
    "log_n_purchases": np.log1p(users["n_purchases"].astype("float32")),
    "log_n_unique_items": np.log1p(users["n_unique_items"].astype("float32")),
    "avg_price": users["avg_price"].astype("float32"),
    "log_days_since_last_purchase": np.log1p(users["days_since_last_purchase"].astype("float32")),
    "is_new_user": users["is_new_user"].astype("float32"),
})
USER_NUM_COLS = list(user_num_raw.columns)
user_num_values, user_num_mean, user_num_std = standardize(user_num_raw)
user_num = np.zeros((N_USERS + 1, len(USER_NUM_COLS)), dtype="float32")
user_num[u] = user_num_values

# ---- item features ----
i = items["item_idx"].to_numpy()
item_cats = np.zeros((N_ITEMS + 1, len(ITEM_CAT_COLS)), dtype="int32")
for column_number, column in enumerate(ITEM_CAT_COLS):
    item_cats[i, column_number] = encode(items[column], vocabularies[column])

item_num_raw = pd.DataFrame({
    "log_purchases_last_7d": np.log1p(items["purchases_last_7d"].astype("float32")),
    "log_purchases_last_28d": np.log1p(items["purchases_last_28d"].astype("float32")),
    "log_purchases_all_time": np.log1p(items["purchases_all_time"].astype("float32")),
    "is_new_item": items["is_new_item"].astype("float32"),
})
ITEM_NUM_COLS = list(item_num_raw.columns)
item_num_values, item_num_mean, item_num_std = standardize(item_num_raw)
item_num = np.zeros((N_ITEMS + 1, len(ITEM_NUM_COLS)), dtype="float32")
item_num[i] = item_num_values

item_article_id = np.full(N_ITEMS + 1, "UNKNOWN", dtype=object)
item_article_id[i] = items["article_id"].to_numpy()

# ---- which IDs did the model actually train on? Others -> 0 (shared "unknown" embedding) ----
user_trained = np.zeros(N_USERS + 1, dtype=bool)
item_trained = np.zeros(N_ITEMS + 1, dtype=bool)
user_trained[train_df["user_idx"].unique()] = True
item_trained[train_df["item_idx"].unique()] = True
user_trained[0] = item_trained[0] = True

# ---- item frequency in training data, for the logQ correction ----
item_counts = np.bincount(train_df["item_idx"], minlength=N_ITEMS + 1).astype("float64")
item_log_q = np.log((item_counts + 1) / (item_counts.sum() + N_ITEMS + 1)).astype("float32")

print("Users with a trained ID embedding:", int(user_trained.sum() - 1))
print("Items with a trained ID embedding:", int(item_trained.sum() - 1))
print("User numeric features:", USER_NUM_COLS)
print("Item numeric features:", ITEM_NUM_COLS)

## 3. Model: Query Tower, Candidate Tower, Two-Tower

Same code as `src/models/query_tower.py`, `src/models/candidate_tower.py` and `src/models/two_tower.py` in the repo.
If you change the model, change it in both places.

In [ ]:
class QueryTower(tf.keras.Model):
    """User side of the two-tower model.

    Inputs (dict of tensors, one row per customer):
      user_idx     int    Week 1 user index (0 = unknown / not seen in training)
      club         int    club_member_status code (0 = unknown)
      news         int    fashion_news_frequency code (0 = unknown)
      month        int    1-12, month of the purchase (context)
      day_of_week  int    1-7, Sunday = 1 (context, same as Spark dayofweek)
      user_num     float  [n] standardized numeric user features
    Output: L2-normalized user embedding of size embedding_dim.
    """

    def __init__(self, n_users, n_club, n_news, embedding_dim=64,
                 id_dim=32, id_dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.id_dropout = id_dropout
        self.user_embedding = tf.keras.layers.Embedding(n_users + 1, id_dim)
        self.club_embedding = tf.keras.layers.Embedding(n_club + 1, 4)
        self.news_embedding = tf.keras.layers.Embedding(n_news + 1, 4)
        self.month_embedding = tf.keras.layers.Embedding(13, 4)
        self.dow_embedding = tf.keras.layers.Embedding(8, 4)
        self.hidden = tf.keras.layers.Dense(128, activation="relu")
        self.dropout = tf.keras.layers.Dropout(0.2)
        self.projection = tf.keras.layers.Dense(embedding_dim)

    def call(self, inputs, training=False):
        user_idx = inputs["user_idx"]
        if training and self.id_dropout > 0:
            # Randomly hide some user IDs (set them to 0) so the shared
            # "unknown user" embedding is trained too. New customers then
            # still get recommendations based on their features.
            hide = tf.random.uniform(tf.shape(user_idx)) < self.id_dropout
            user_idx = tf.where(hide, tf.zeros_like(user_idx), user_idx)

        x = tf.concat([
            self.user_embedding(user_idx),
            self.club_embedding(inputs["club"]),
            self.news_embedding(inputs["news"]),
            self.month_embedding(inputs["month"]),
            self.dow_embedding(inputs["day_of_week"]),
            tf.cast(inputs["user_num"], tf.float32),
        ], axis=1)
        x = self.dropout(self.hidden(x), training=training)
        return tf.math.l2_normalize(self.projection(x), axis=1)


class CandidateTower(tf.keras.Model):
    """Item side of the two-tower model.

    Inputs (dict of tensors, one row per article):
      item_idx  int    Week 1 item index (0 = unknown / not seen in training)
      cats      int    [4] category codes: product_type, colour_group,
                       department, garment_group (0 = unknown)
      item_num  float  [n] standardized numeric item features (popularity)
    Output: L2-normalized item embedding of size embedding_dim.
    """

    def __init__(self, n_items, cat_sizes, embedding_dim=64,
                 id_dim=32, id_dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.id_dropout = id_dropout
        self.item_embedding = tf.keras.layers.Embedding(n_items + 1, id_dim)
        self.cat_embeddings = [
            tf.keras.layers.Embedding(size + 1, 8) for size in cat_sizes
        ]
        self.hidden = tf.keras.layers.Dense(128, activation="relu")
        self.dropout = tf.keras.layers.Dropout(0.2)
        self.projection = tf.keras.layers.Dense(embedding_dim)

    def call(self, inputs, training=False):
        item_idx = inputs["item_idx"]
        if training and self.id_dropout > 0:
            # Same idea as the query tower: train the "unknown item" embedding
            # so brand-new articles can still be recommended from metadata.
            hide = tf.random.uniform(tf.shape(item_idx)) < self.id_dropout
            item_idx = tf.where(hide, tf.zeros_like(item_idx), item_idx)

        parts = [self.item_embedding(item_idx)]
        for column, embedding in enumerate(self.cat_embeddings):
            parts.append(embedding(inputs["cats"][:, column]))
        parts.append(tf.cast(inputs["item_num"], tf.float32))

        x = tf.concat(parts, axis=1)
        x = self.dropout(self.hidden(x), training=training)
        return tf.math.l2_normalize(self.projection(x), axis=1)


class TwoTowerModel(tf.keras.Model):
    """Combines the two towers and holds the training loss."""

    def __init__(self, query_tower, candidate_tower, temperature=0.05, **kwargs):
        super().__init__(**kwargs)
        self.query_tower = query_tower
        self.candidate_tower = candidate_tower
        self.temperature = temperature

    def call(self, inputs, training=False):
        user_vectors = self.query_tower(inputs["query"], training=training)
        item_vectors = self.candidate_tower(inputs["candidate"], training=training)
        # cosine similarity of every user with every item in the batch
        return tf.matmul(user_vectors, item_vectors, transpose_b=True) / self.temperature

    def in_batch_loss(self, batch, training=False):
        """Softmax loss with in-batch negative sampling.

        Row i of the batch is (customer i, article i they bought). For
        customer i, article i is the positive and the other articles in the
        batch are the negatives.
        - Duplicates of the positive article, and other articles bought by the
          same customer, are ignored (they are not real negatives).
        - logQ correction: popular articles appear as negatives much more
          often, so their log-frequency is subtracted to avoid punishing them.
        """
        logits = self(
            {"query": batch["query"], "candidate": batch["candidate"]},
            training=training,
        )
        logits = logits - batch["log_q"][None, :]

        user_keys = batch["user_key"]
        item_keys = batch["item_key"]
        n = tf.shape(logits)[0]
        is_positive = tf.eye(n, dtype=tf.bool)
        same_item = tf.equal(item_keys[:, None], item_keys[None, :])
        same_user = tf.equal(user_keys[:, None], user_keys[None, :])
        ignore = tf.logical_and(
            tf.logical_or(same_item, same_user), tf.logical_not(is_positive)
        )
        logits = tf.where(ignore, tf.constant(-1e9, logits.dtype), logits)

        loss = tf.nn.softmax_cross_entropy_with_logits(
            labels=tf.cast(is_positive, logits.dtype), logits=logits
        )
        return tf.reduce_mean(loss)

## 4. TensorFlow datasets

In [ ]:
def model_user_id(user_idx):
    return np.where(user_trained[user_idx], user_idx, 0).astype("int32")


def model_item_id(item_idx):
    return np.where(item_trained[item_idx], item_idx, 0).astype("int32")


def query_features(user_idx, month, day_of_week):
    return {
        "user_idx": model_user_id(user_idx),
        "club": user_club[user_idx],
        "news": user_news[user_idx],
        "month": np.asarray(month, dtype="int32"),
        "day_of_week": np.asarray(day_of_week, dtype="int32"),
        "user_num": user_num[user_idx],
    }


def candidate_features(item_idx):
    return {
        "item_idx": model_item_id(item_idx),
        "cats": item_cats[item_idx],
        "item_num": item_num[item_idx],
    }


def to_tensors(features):
    return {name: tf.convert_to_tensor(value) for name, value in features.items()}


def make_dataset(frame, shuffle):
    user_idx = frame["user_idx"].to_numpy()
    item_idx = frame["item_idx"].to_numpy()
    data = {
        "query": query_features(user_idx, frame["month"], frame["day_of_week"]),
        "candidate": candidate_features(item_idx),
        "user_key": user_idx.astype("int32"),
        "item_key": item_idx.astype("int32"),
        "log_q": item_log_q[item_idx],
    }
    dataset = tf.data.Dataset.from_tensor_slices(data)
    if shuffle:
        dataset = dataset.shuffle(100_000, seed=SEED, reshuffle_each_iteration=True)
    return dataset.batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)

query_tower = QueryTower(
    N_USERS, len(vocabularies["club_member_status"]), len(vocabularies["fashion_news_frequency"]),
    EMBEDDING_DIM, ID_DIM, ID_DROPOUT,
)
candidate_tower = CandidateTower(
    N_ITEMS, [len(vocabularies[column]) for column in ITEM_CAT_COLS],
    EMBEDDING_DIM, ID_DIM, ID_DROPOUT,
)
model = TwoTowerModel(query_tower, candidate_tower, TEMPERATURE)

sample_batch = next(iter(val_ds))
initial_loss = float(model.in_batch_loss(sample_batch, training=False))
print("Loss before training:", round(initial_loss, 3))
print("Random-guessing loss is about:", round(float(np.log(BATCH_SIZE)), 3))

## 5. Train with early stopping
Training loss should go clearly **below** the random-guessing value printed above.

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)


@tf.function
def train_step(batch):
    with tf.GradientTape() as tape:
        loss = model.in_batch_loss(batch, training=True)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss


@tf.function
def val_step(batch):
    return model.in_batch_loss(batch, training=False)


history = {"train_loss": [], "val_loss": []}
best_val_loss, best_weights, best_epoch, bad_epochs = np.inf, None, 0, 0

for epoch in range(1, EPOCHS + 1):
    started = time.time()
    train_mean, val_mean = tf.keras.metrics.Mean(), tf.keras.metrics.Mean()
    for batch in train_ds:
        train_mean.update_state(train_step(batch))
    for batch in val_ds:
        val_mean.update_state(val_step(batch))

    train_loss, val_loss = float(train_mean.result()), float(val_mean.result())
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch}/{EPOCHS} | train loss {train_loss:.4f} | "
          f"val loss {val_loss:.4f} | {time.time() - started:.0f}s")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss, best_weights, best_epoch, bad_epochs = val_loss, model.get_weights(), epoch, 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("Early stopping: validation loss stopped improving.")
            break

model.set_weights(best_weights)
print(f"Using weights from epoch {best_epoch} (val loss {best_val_loss:.4f})")

In [ ]:
epochs_run = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs_run, history["train_loss"], marker="o", label="Training loss")
plt.plot(epochs_run, history["val_loss"], marker="o", label="Validation loss")
plt.axhline(np.log(BATCH_SIZE), color="grey", linestyle="--", label="Random guessing")
plt.xlabel("Epoch")
plt.ylabel("In-batch softmax loss")
plt.title("Two-Tower training history")
plt.xticks(list(epochs_run))
plt.grid(True)
plt.legend()
plt.savefig(os.path.join(ARTIFACT_DIR, "loss_curve.png"), dpi=120, bbox_inches="tight")
plt.show()

## 6. Evaluate on the test week (2020-09-16 to 2020-09-22)

Every test customer gets 12 recommendations from the **full catalogue** (all articles).
We compare with the **popularity baseline**: the same 12 most popular articles (`fallback_top12.json`) for everyone.

Items a customer bought before are **not** removed: H&M customers often buy the same article again.

In [ ]:
# embeddings for every article in the catalogue (row r = item_idx r + 1)
catalog_item_idx = np.arange(1, N_ITEMS + 1, dtype="int32")
catalog_parts = []
for start in range(0, N_ITEMS, 8192):
    part = catalog_item_idx[start:start + 8192]
    catalog_parts.append(candidate_tower(to_tensors(candidate_features(part)), training=False))
catalog_embeddings = tf.concat(catalog_parts, axis=0)
print("Catalogue embeddings:", catalog_embeddings.shape)

# context for recommending in the test week = the cutoff date
context_month = CUTOFF.month
context_day_of_week = (CUTOFF.day_of_week + 1) % 7 + 1   # Spark style: Sunday = 1

ground_truth = test_df.groupby("user_idx")["item_idx"].apply(set)
eval_users = ground_truth.index.to_numpy().astype("int32")
print("Test customers:", len(eval_users),
      "| of which new (no trained ID):", int((~user_trained[eval_users]).sum()))

In [ ]:
def recommend(user_idx, k=TOP_K, batch_size=1024):
    """Top-k item_idx for each user, scored against the full catalogue."""
    results = []
    for start in range(0, len(user_idx), batch_size):
        batch_users = user_idx[start:start + batch_size]
        n = len(batch_users)
        features = query_features(batch_users, np.full(n, context_month), np.full(n, context_day_of_week))
        user_vectors = query_tower(to_tensors(features), training=False)
        scores = tf.matmul(user_vectors, catalog_embeddings, transpose_b=True)
        results.append(tf.math.top_k(scores, k=k).indices.numpy() + 1)
    return np.concatenate(results)


def evaluate(recommendations, users_to_score):
    totals = {name: 0.0 for name in
              ["Recall@5", "Recall@10", "Recall@12", "NDCG@5", "NDCG@10", "NDCG@12", "MAP@12"]}
    for user, recs in zip(users_to_score, recommendations):
        relevant = ground_truth[user]
        hits = np.array([item in relevant for item in recs[:TOP_K]], dtype="float64")
        for k in (5, 10, 12):
            totals[f"Recall@{k}"] += hits[:k].sum() / len(relevant)
            dcg = (hits[:k] / np.log2(np.arange(2, k + 2))).sum()
            idcg = (1.0 / np.log2(np.arange(2, min(len(relevant), k) + 2))).sum()
            totals[f"NDCG@{k}"] += dcg / idcg
        precision_at_i = np.cumsum(hits) / np.arange(1, TOP_K + 1)
        totals["MAP@12"] += (precision_at_i * hits).sum() / min(len(relevant), TOP_K)
    return {name: value / len(users_to_score) for name, value in totals.items()}


model_recs = recommend(eval_users)
model_metrics = evaluate(model_recs, eval_users)

article_to_item = dict(zip(items["article_id"], items["item_idx"]))
baseline_items = np.array([article_to_item[a] for a in fallback_top12 if a in article_to_item])
baseline_metrics = evaluate([baseline_items] * len(eval_users), eval_users)

results = pd.DataFrame({"Two-tower model": model_metrics, "Popularity baseline": baseline_metrics})
results["Model / baseline"] = results["Two-tower model"] / results["Popularity baseline"].replace(0, np.nan)
print(f"Random recommendations would give Recall@12 of about {TOP_K / N_ITEMS:.5f}")
display(results.round(5))

### How to read the results
- **Model vs random:** the model should be far above random (shown above the table).
- **Model vs popularity baseline:** "most popular items" is a strong baseline for H&M. If the model is below it, try:
  1. `TRAIN_WEEKS = 6` (fashion changes fast, so recent data matters most)
  2. `EPOCHS = 15` (if validation loss was still dropping at the end)
  3. `TEMPERATURE = 0.1`

  Then re-run all cells and write the best result in the report.

In [ ]:
# sample recommendations for one test customer
sample_user = int(eval_users[user_trained[eval_users]][0]) if user_trained[eval_users].any() else int(eval_users[0])
sample_recs = recommend(np.array([sample_user]), k=10)[0]
sample_table = items.set_index("item_idx").loc[sample_recs, ["article_id"] + ITEM_CAT_COLS].reset_index(drop=True)
sample_table.insert(0, "recommendation_rank", range(1, len(sample_table) + 1))
sample_table["bought_in_test_week"] = [item in ground_truth[sample_user] for item in sample_recs]
print("Customer user_idx:", sample_user)
display(sample_table)

## 7. Save the model and the results

- **Google Drive** `hm_project/week2_model/`: model weights, item embeddings and config, for Week 3 (Redis + ANN). Too big for GitHub.
- **Download** `week2_artifacts.zip`: small result files for the repo's `artifacts/` folder.

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)
query_tower.save_weights(os.path.join(MODEL_DIR, "query_tower.weights.h5"))
candidate_tower.save_weights(os.path.join(MODEL_DIR, "candidate_tower.weights.h5"))
np.save(os.path.join(MODEL_DIR, "item_embeddings.npy"), catalog_embeddings.numpy())
with open(os.path.join(MODEL_DIR, "item_ids.json"), "w") as f:
    json.dump(item_article_id[1:].tolist(), f)            # row r of item_embeddings = item_ids[r]
np.save(os.path.join(MODEL_DIR, "trained_user_idx.npy"), np.flatnonzero(user_trained[1:]) + 1)

model_config = {
    "architecture": "Context-aware two-tower retrieval model (TensorFlow / Keras)",
    "embedding_dim": EMBEDDING_DIM, "id_dim": ID_DIM, "temperature": TEMPERATURE,
    "id_dropout": ID_DROPOUT, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
    "loss": "in-batch softmax with logQ correction",
    "epochs_run": len(history["train_loss"]), "best_epoch": best_epoch,
    "best_val_loss": best_val_loss,
    "cutoff": str(CUTOFF.date()), "train_weeks": TRAIN_WEEKS,
    "training_rows": len(train_df), "validation_rows": len(val_df), "test_rows": len(test_df),
    "n_users": N_USERS, "n_items": N_ITEMS,
    "n_club": len(vocabularies["club_member_status"]),
    "n_news": len(vocabularies["fashion_news_frequency"]),
    "item_cat_cols": ITEM_CAT_COLS,
    "item_cat_sizes": [len(vocabularies[column]) for column in ITEM_CAT_COLS],
    "user_num_cols": USER_NUM_COLS,
    "user_num_mean": {k: float(v) for k, v in user_num_mean.items()},
    "user_num_std": {k: float(v) for k, v in user_num_std.items()},
    "item_num_cols": ITEM_NUM_COLS,
    "item_num_mean": {k: float(v) for k, v in item_num_mean.items()},
    "item_num_std": {k: float(v) for k, v in item_num_std.items()},
    "category_codes": "position in vocabularies.json + 1; 0 = unknown",
    "unknown_id_rule": "user_idx not in trained_user_idx.npy -> 0; items not seen in training -> 0",
    "context": "month and day_of_week (Sunday = 1) of the recommendation date",
}
with open(os.path.join(MODEL_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)

print("Saved to Drive:", MODEL_DIR)
for name in sorted(os.listdir(MODEL_DIR)):
    print(f"  {name}: {os.path.getsize(os.path.join(MODEL_DIR, name)) / 1e6:.1f} MB")

In [ ]:
with open(os.path.join(ARTIFACT_DIR, "week2_evaluation_metrics.json"), "w") as f:
    json.dump({
        "test_period": "2020-09-16 to 2020-09-22",
        "test_customers": int(len(eval_users)),
        "catalogue_articles": int(N_ITEMS),
        "two_tower_model": model_metrics,
        "popularity_baseline": baseline_metrics,
    }, f, indent=2)
with open(os.path.join(ARTIFACT_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=2)
with open(os.path.join(ARTIFACT_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)
sample_table.to_csv(os.path.join(ARTIFACT_DIR, "sample_top10_recommendations.csv"), index=False)

metric_rows = "\n".join(
    f"| {name} | {model_metrics[name]:.5f} | {baseline_metrics[name]:.5f} |" for name in model_metrics
)
report = f"""# Week 2 — Deep Learning Model Architecture

## Implemented
- Query Tower: customer ID, club status, news frequency, age, purchase count, unique items,
  average price, recency, new-customer flag + context (month, day of week)
- Candidate Tower: article ID, product type, colour group, department, garment group,
  7-day / 28-day / all-time popularity, new-article flag
- L2-normalised embeddings, cosine similarity with temperature {TEMPERATURE}
- In-batch negative sampling (softmax loss) with logQ popularity correction
- Cold start: IDs not seen in training share a trained "unknown" embedding (index 0)
- Early stopping on a validation week

## Data (Week 1 outputs)
- Training: {len(train_df):,} purchases ({TRAIN_WEEKS} weeks before 2020-09-16, minus the last week)
- Validation: {len(val_df):,} purchases (2020-09-09 to 2020-09-15)
- Test: {len(test_df):,} purchases by {len(eval_users):,} customers (2020-09-16 to 2020-09-22)
- Catalogue: {N_ITEMS:,} articles (full catalogue scored for every customer)

## Training
- Adam, learning rate {LEARNING_RATE}, batch size {BATCH_SIZE}, embedding size {EMBEDDING_DIM}
- Epochs run: {len(history["train_loss"])}, best epoch: {best_epoch}, best validation loss: {best_val_loss:.4f}
- Random-guessing loss: {np.log(BATCH_SIZE):.4f}

## Test-week results (12 recommendations per customer)
| Metric | Two-tower model | Popularity baseline |
|---|---|---|
{metric_rows}

## Model files (Google Drive, `hm_project/week2_model/`)
`query_tower.weights.h5`, `candidate_tower.weights.h5`, `item_embeddings.npy`, `item_ids.json`,
`trained_user_idx.npy`, `model_config.json`
"""
with open(os.path.join(ARTIFACT_DIR, "WEEK2_REPORT.md"), "w", encoding="utf-8") as f:
    f.write(report)

shutil.copytree(ARTIFACT_DIR, os.path.join(DATA_PATH, "week2_artifacts"), dirs_exist_ok=True)
shutil.make_archive(ARTIFACT_DIR, "zip", ARTIFACT_DIR)
print(report)
print("Artifacts:", sorted(os.listdir(ARTIFACT_DIR)))

In [ ]:
from google.colab import files
files.download("/content/week2_artifacts.zip")   # put these files in the repo's artifacts/ folder

## Week 2 checklist
- [x] Query Tower with user features and purchase context
- [x] Candidate Tower with article metadata and popularity
- [x] In-batch negative sampling (with logQ correction)
- [x] Time-based train / validation / test split using Week 1 outputs
- [x] Early stopping on validation loss
- [x] Recall@K, NDCG@K and MAP@12 on the full catalogue, compared with a popularity baseline
- [x] Model weights, item embeddings and config saved to Drive for Week 3

**Note for Week 3:** load the towers with the settings in `model_config.json`, call each tower once on a
sample input to build it, then `load_weights(...)`. The item vectors for Redis / ANN are in `item_embeddings.npy`
(row order = `item_ids.json`).